# TAC / EDDIT — revised Toronto venue visits, repeat visitors, and home origins

This notebook rebuilds the Toronto Arts Council mobility outputs for **July 1, 2023 through June 30, 2026**.

## Principal revisions

- Uses the revised venue boundaries at `./venues-boundaries.geo.json` for local validation.
- Uses the precomputed venue lookup at `./venues_geohash9.csv` for Snowflake matching.
- Matches venue activity directly at **geohash level 9**.
- Keeps the existing Toronto geohash8 cover for Toronto-wide comparison denominators.
- Keeps the existing Toronto geohash6 cover for Toronto HOME filtering.
- Excludes source stops whose full `DWELL_TIME_MINUTES` exceeds eight hours.
- Merges nearby/fragmented stops for the same device and venue into visits using a configurable gap.
- Measures repeat attendance within fixed calendar half-years, avoiding a multi-year unique-person interpretation.
- Writes all final outputs to Snowflake and exports matching CSVs to `./activity/`.

## Visit construction

For each venue and device, source stops are ordered by `STOP_ZONED_DATETIME`. The stop end is derived as:

```text
STOP_ZONED_DATETIME + DWELL_TIME_MINUTES
```

A new visit starts when the next stop begins more than `VISIT_MERGE_GAP_MINUTES` after the latest prior stop end. This logic operates continuously across midnight. A merged visit is assigned to the date, month, half-year, and time periods of its first stop.

## Repeat-visitor definition

A repeat visitor is a device observed at the same venue on at least **two distinct visit dates within the same half-year**. The six analysis periods are `2023H2` through `2026H1`.

## Output tables

- `DEDICATED.EDDIT_TAC.venues_stops`
- `DEDICATED.EDDIT_TAC.venues_repeat_visitors`
- `DEDICATED.EDDIT_TAC.venues_homes_monthly`
- `DEDICATED.EDDIT_TAC.venues_homes`
- `DEDICATED.EDDIT_TAC.toronto_month_denoms`
- `DEDICATED.EDDIT_TAC.venues_geohash9_diagnostics`

Persistent intermediate tables are written under `DEDICATED.EDDIT_TAC_TMP` so long-running work survives notebook or browser interruptions.

## 1. Imports and Snowflake connection

In [1]:
import time
from contextlib import contextmanager
from pathlib import Path

import pandas as pd
import geopandas as gpd
from IPython.display import display

try:
    from tqdm.notebook import tqdm
except ImportError:
    !pip install tqdm -q
    from tqdm.notebook import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

In [3]:
%load_ext cuebiqmagic.magics
%init_cuebiq_data
snow_engine = get_ipython().user_ns["instance"]
print("Snowflake connection initialized.")

The cuebiqmagic.magics extension is already loaded. To reload it, use:
  %reload_ext cuebiqmagic.magics
schema : args.schema='paas_cda_pe_v3' and database: args.database='cuebiq_data' used
The connection to the Snowflake cluster has been established using database 'cuebiq_data' and schema 'paas_cda_pe_v3'.

You can now retrieve the connection using the following statement:
snow_engine = get_ipython().user_ns['instance']

The connection offers two methods


snow_engine.execute_statement("<SQL Statement>") # execute the SQL statement.


snow_engine.read_sql("<SQL Statement>") # execute the SQL statement and return a Pandas DataFrame.


snow_engine.write_dataframe(dataframe:pd.DataFrame, table_name:str,schema:str, index_in=False, if_exists_in="append",method_in="multi")
Snowflake connection initialized.


## 2. Constants

In [4]:
# -------------------------------------------------------------------
# Source and destination objects
# -------------------------------------------------------------------
SCHEMA_NAME = {"cda": "CUEBIQ_DATA.PAAS_CDA_PE_V3"}

OUT_SCHEMA = "DEDICATED.EDDIT_TAC"
TMP_SCHEMA = "DEDICATED.EDDIT_TAC_TMP"

STOP_BY_EVENT_TABLE = f"{SCHEMA_NAME['cda']}.STOP_BY_EVENT_DATE_UPLEVELLED"
DEVICE_RECURRING_AREA_TABLE = f"{SCHEMA_NAME['cda']}.DEVICE_RECURRING_AREA"

OUT_TABLE_VENUES_STOPS = f"{OUT_SCHEMA}.venues_stops"
OUT_TABLE_VENUES_REPEAT = f"{OUT_SCHEMA}.venues_repeat_visitors"
OUT_TABLE_VENUES_HOMES_MONTHLY = f"{OUT_SCHEMA}.venues_homes_monthly"
OUT_TABLE_VENUES_HOMES = f"{OUT_SCHEMA}.venues_homes"
OUT_TABLE_TORONTO_DENOMS = f"{OUT_SCHEMA}.toronto_month_denoms"
OUT_TABLE_GEOHASH_DIAGNOSTICS = f"{OUT_SCHEMA}.venues_geohash9_diagnostics"

# -------------------------------------------------------------------
# Study parameters
# -------------------------------------------------------------------
COUNTRY_CODE = "CA"
PROVIDER_ID = "PAPA"

DATE_START = 20230701
DATE_END = 20260630

PILOT_DATE_START = 20230701
PILOT_DATE_END = 20230930
PILOT_VENUES = [
    "Meridian Arts Centre",
    "Four Seasons Centre for the Performing Arts",
    "Lakeshore Arts",
]

HOME_CONFIDENCE_MIN = 0.60
MAX_RAW_STOP_MINUTES = 8 * 60
MAX_MERGED_VISIT_MINUTES = 8 * 60
VISIT_MERGE_GAP_MINUTES = 60

TIME_PERIODS = ["all", "weekdays", "weekends", "nine-five", "evening"]

# -------------------------------------------------------------------
# Local files
# -------------------------------------------------------------------
VENUES_PATH = "./venues-boundaries.geo.json"
VENUE_GEOHASH9_CSV = "./venues_geohash9.csv"
TORONTO_GEOHASH6_CSV = "./geohash/toronto_geohash6.csv"
TORONTO_GEOHASH8_CSV = "./geohash/toronto_geohash8.csv"
EXPORT_DIR = Path("./activity")

# -------------------------------------------------------------------
# Lookup tables
# -------------------------------------------------------------------
TMP_TORONTO_GEOHASH8 = f"{TMP_SCHEMA}.TMP_TAC_TORONTO_GEOHASH8"
TMP_TORONTO_GEOHASH6 = f"{TMP_SCHEMA}.TMP_TAC_TORONTO_GEOHASH6"
TMP_VENUE_GEOHASH9 = f"{TMP_SCHEMA}.TMP_TAC_VENUE_GEOHASH9"

# -------------------------------------------------------------------
# Full-run working tables
# -------------------------------------------------------------------
TMP_TORONTO_STOP_ROWS = f"{TMP_SCHEMA}.TMP_TAC_TORONTO_STOP_ROWS"
TMP_TORONTO_MONTH_DENOMS = f"{TMP_SCHEMA}.TMP_TAC_TORONTO_MONTH_DENOMS"
TMP_VENUE_STOP_ROWS = f"{TMP_SCHEMA}.TMP_TAC_VENUE_STOP_ROWS"
TMP_VENUE_VISITS = f"{TMP_SCHEMA}.TMP_TAC_VENUE_VISITS"
TMP_VENUE_DEVICE_MONTH_PERIOD = f"{TMP_SCHEMA}.TMP_TAC_VENUE_DEVICE_MONTH_PERIOD"
TMP_VENUE_DEVICE_HALF_PERIOD = f"{TMP_SCHEMA}.TMP_TAC_VENUE_DEVICE_HALF_PERIOD"
TMP_HOME_MONTH = f"{TMP_SCHEMA}.TMP_TAC_HOME_MONTH"
TMP_VENUE_DEVICE_HOME_MONTH_PERIOD = f"{TMP_SCHEMA}.TMP_TAC_VENUE_DEVICE_HOME_MONTH_PERIOD"

# Pilot tables use a separate prefix and do not overwrite full-run tables.
PILOT_PREFIX = f"{TMP_SCHEMA}.TMP_TAC_PILOT"
PILOT_TORONTO_STOP_ROWS = f"{PILOT_PREFIX}_TORONTO_STOP_ROWS"
PILOT_TORONTO_DENOMS = f"{PILOT_PREFIX}_TORONTO_MONTH_DENOMS"
PILOT_VENUE_STOP_ROWS = f"{PILOT_PREFIX}_VENUE_STOP_ROWS"
PILOT_VENUE_VISITS = f"{PILOT_PREFIX}_VENUE_VISITS"
PILOT_DEVICE_MONTH_PERIOD = f"{PILOT_PREFIX}_DEVICE_MONTH_PERIOD"
PILOT_DEVICE_HALF_PERIOD = f"{PILOT_PREFIX}_DEVICE_HALF_PERIOD"
PILOT_HOME_MONTH = f"{PILOT_PREFIX}_HOME_MONTH"
PILOT_DEVICE_HOME_MONTH_PERIOD = f"{PILOT_PREFIX}_DEVICE_HOME_MONTH_PERIOD"

print(f"Study period: {DATE_START}–{DATE_END}")
print(f"Pilot period: {PILOT_DATE_START}–{PILOT_DATE_END}")
print(f"Visit merge gap: {VISIT_MERGE_GAP_MINUTES} minutes")

Study period: 20230701–20260630
Pilot period: 20230701–20230930
Visit merge gap: 60 minutes


## 3. Helpers

In [5]:
@contextmanager
def timer(label: str):
    start = time.time()
    print(f"[START] {label}")
    try:
        yield
    except Exception:
        elapsed = time.time() - start
        print(f"[FAIL]  {label} after {elapsed:,.2f}s")
        raise
    else:
        elapsed = time.time() - start
        print(f"[DONE]  {label} ({elapsed:,.2f}s)")


def sql_str_list(values):
    escaped = [str(v).replace("'", "''") for v in values]
    return "(" + ",".join(f"'{v}'" for v in escaped) + ")"


def upload_lookup_table(df, table_name, column_types, chunk_size=5000):
    """Recreate a small Snowflake lookup table and insert rows in chunks."""
    if df.empty:
        raise ValueError(f"DataFrame for {table_name} is empty.")

    cols = [c for c, _ in column_types]
    missing = sorted(set(cols) - set(df.columns))
    if missing:
        raise ValueError(f"Missing upload columns for {table_name}: {missing}")

    create_cols_sql = ", ".join(f"{c} {t}" for c, t in column_types)
    snow_engine.read_sql(f"CREATE OR REPLACE TABLE {table_name} ({create_cols_sql})")

    for start in tqdm(range(0, len(df), chunk_size), desc=f"Uploading {table_name.split('.')[-1]}"):
        chunk = df.iloc[start:start + chunk_size]
        value_rows = []
        for row in chunk[cols].itertuples(index=False, name=None):
            vals = []
            for value in row:
                if pd.isna(value):
                    vals.append("NULL")
                elif isinstance(value, (int, float)) and not isinstance(value, bool):
                    vals.append(str(value))
                elif isinstance(value, bool):
                    vals.append("TRUE" if value else "FALSE")
                else:
                    escaped = str(value).replace("'", "''")
                    vals.append(f"'{escaped}'")
            value_rows.append("(" + ", ".join(vals) + ")")

        insert_sql = f"""
        INSERT INTO {table_name} ({', '.join(cols)})
        VALUES {', '.join(value_rows)}
        """
        snow_engine.read_sql(insert_sql)

    loaded_n = int(snow_engine.read_sql(f"SELECT COUNT(*) AS n FROM {table_name}").iloc[0, 0])
    if loaded_n != len(df):
        raise RuntimeError(f"Expected {len(df):,} rows in {table_name}, found {loaded_n:,}.")
    print(f"Loaded {loaded_n:,} rows into {table_name}")


def export_table_to_csv(table_name, out_path, order_by=None):
    sql = f"SELECT * FROM {table_name}"
    if order_by:
        sql += f" ORDER BY {order_by}"
    with timer(f"Read {table_name}"):
        df = snow_engine.read_sql(sql)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"Exported {len(df):,} rows -> {out_path}")
    return df

## 4. Create schemas and verify source coverage

In [6]:
with timer("Ensure output and working schemas exist"):
    snow_engine.read_sql(f"CREATE SCHEMA IF NOT EXISTS {OUT_SCHEMA}")
    snow_engine.read_sql(f"CREATE SCHEMA IF NOT EXISTS {TMP_SCHEMA}")

coverage_sql = f"""
SELECT
    'stops' AS source,
    MIN(event_date) AS min_date,
    MAX(event_date) AS max_date,
    COUNT_IF(event_date BETWEEN {DATE_START} AND {DATE_END}) AS rows_in_requested_period
FROM {STOP_BY_EVENT_TABLE}
WHERE provider_id = '{PROVIDER_ID}'
  AND country_code = '{COUNTRY_CODE}'
  AND event_date BETWEEN {DATE_START} AND {DATE_END}

UNION ALL

SELECT
    'home' AS source,
    MIN(snapshot_event_date) AS min_date,
    MAX(snapshot_event_date) AS max_date,
    COUNT_IF(snapshot_event_date BETWEEN {DATE_START} AND {DATE_END}) AS rows_in_requested_period
FROM {DEVICE_RECURRING_AREA_TABLE}
WHERE provider_id = '{PROVIDER_ID}'
  AND country_code = '{COUNTRY_CODE}'
  AND snapshot_event_date BETWEEN {DATE_START} AND {DATE_END}
"""

with timer("Check requested source coverage"):
    source_coverage_df = snow_engine.read_sql(coverage_sql)
display(source_coverage_df)

if source_coverage_df["max_date"].min() < DATE_END:
    print("WARNING: At least one source does not reach the requested end date.")

[START] Ensure output and working schemas exist
[DONE]  Ensure output and working schemas exist (1.13s)
[START] Check requested source coverage
[DONE]  Check requested source coverage (1,819.09s)


,source,min_date,max_date,rows_in_requested_period
0,stops,20230701,20260630,5482052771
1,home,20230701,20260630,38670915873


## 5. Load and validate venue and Toronto lookup files

In [7]:
required_paths = [
    VENUES_PATH,
    VENUE_GEOHASH9_CSV,
    TORONTO_GEOHASH6_CSV,
    TORONTO_GEOHASH8_CSV,
]
missing_paths = [path for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required local files: {missing_paths}")

with timer("Load local geography files"):
    venues_gdf = gpd.read_file(VENUES_PATH)
    venue_geohash9_df = pd.read_csv(
        VENUE_GEOHASH9_CSV,
        dtype={"venue_id": "Int64", "venue_name": "string", "geohash9": "string"},
    )
    toronto8_df = pd.read_csv(TORONTO_GEOHASH8_CSV, dtype={"geohash": "string"})
    toronto6_df = pd.read_csv(TORONTO_GEOHASH6_CSV, dtype={"geohash": "string"})

print(f"Boundary features: {len(venues_gdf):,}")
print(f"Venue-geohash9 rows: {len(venue_geohash9_df):,}")

[START] Load local geography files
[DONE]  Load local geography files (0.72s)
Boundary features: 85
Venue-geohash9 rows: 9,314


In [8]:
# Normalize lookup files.
required_venue_cols = {"venue_id", "venue_name", "geohash9", "coverage_fraction"}
missing = required_venue_cols - set(venue_geohash9_df.columns)
if missing:
    raise ValueError(f"{VENUE_GEOHASH9_CSV} is missing columns: {sorted(missing)}")

venue_geohash9_df = venue_geohash9_df[list(required_venue_cols)].copy()
venue_geohash9_df["venue_name"] = venue_geohash9_df["venue_name"].str.strip()
venue_geohash9_df["geohash9"] = venue_geohash9_df["geohash9"].str.strip().str.lower()
venue_geohash9_df["coverage_fraction"] = pd.to_numeric(
    venue_geohash9_df["coverage_fraction"], errors="coerce"
)
venue_geohash9_df = (
    venue_geohash9_df
    .dropna(subset=["venue_id", "venue_name", "geohash9", "coverage_fraction"])
    .query("venue_name != '' and geohash9 != ''")
    .drop_duplicates(["venue_id", "venue_name", "geohash9"])
    .sort_values(["venue_id", "venue_name", "geohash9"])
    .reset_index(drop=True)
)

for label, df, precision in [
    ("venue", venue_geohash9_df, 9),
]:
    invalid = df[df[f"geohash{precision}"].str.len() != precision]
    if not invalid.empty:
        raise ValueError(f"Found {len(invalid):,} invalid {label} geohash lengths.")

if not venue_geohash9_df["coverage_fraction"].between(0, 1).all():
    raise ValueError("coverage_fraction must be between 0 and 1.")

if "geohash" not in toronto8_df or "geohash" not in toronto6_df:
    raise ValueError("Toronto cover CSVs must each contain a 'geohash' column.")

toronto8_df = (
    toronto8_df[["geohash"]].dropna()
    .assign(geohash=lambda d: d["geohash"].str.strip().str.lower())
    .query("geohash != ''")
    .drop_duplicates().rename(columns={"geohash": "geohash8"})
    .sort_values("geohash8").reset_index(drop=True)
)
toronto6_df = (
    toronto6_df[["geohash"]].dropna()
    .assign(geohash=lambda d: d["geohash"].str.strip().str.lower())
    .query("geohash != ''")
    .drop_duplicates().rename(columns={"geohash": "geohash6"})
    .sort_values("geohash6").reset_index(drop=True)
)

if not toronto8_df["geohash8"].str.len().eq(8).all():
    raise ValueError("Toronto geohash8 file contains values not exactly 8 characters long.")
if not toronto6_df["geohash6"].str.len().eq(6).all():
    raise ValueError("Toronto geohash6 file contains values not exactly 6 characters long.")

print(f"Unique venues in geohash lookup: {venue_geohash9_df['venue_id'].nunique():,}")
print(f"Toronto geohash8 rows: {len(toronto8_df):,}")
print(f"Toronto geohash6 rows: {len(toronto6_df):,}")

Unique venues in geohash lookup: 85
Toronto geohash8 rows: 1,210,820
Toronto geohash6 rows: 1,307


In [9]:
# Boundary-file validation.
#
# The GeoJSON uses "id" for the venue identifier, whereas the geohash CSV
# uses "venue_id". Normalize both to a shared string-valued "venue_id" field.

venues_gdf = venues_gdf.copy()

required_boundary_cols = {"venue_name", "geometry"}
missing_boundary_cols = required_boundary_cols - set(venues_gdf.columns)

if missing_boundary_cols:
    raise ValueError(
        f"{VENUES_PATH} is missing required columns: "
        f"{sorted(missing_boundary_cols)}"
    )

# Normalize the identifier column.
if "venue_id" in venues_gdf.columns:
    venues_gdf["venue_id"] = venues_gdf["venue_id"]
elif "id" in venues_gdf.columns:
    venues_gdf["venue_id"] = venues_gdf["id"]
elif "fid" in venues_gdf.columns:
    venues_gdf["venue_id"] = venues_gdf["fid"]
else:
    raise ValueError(
        f"{VENUES_PATH} has no venue identifier column. "
        "Expected one of: venue_id, id, fid."
    )

# Normalize identifiers and names before comparison.
venues_gdf["venue_id"] = (
    venues_gdf["venue_id"]
    .astype("string")
    .str.strip()
)

venues_gdf["venue_name"] = (
    venues_gdf["venue_name"]
    .astype("string")
    .str.strip()
)

venue_geohash9_df["venue_id"] = (
    venue_geohash9_df["venue_id"]
    .astype("string")
    .str.strip()
)

venue_geohash9_df["venue_name"] = (
    venue_geohash9_df["venue_name"]
    .astype("string")
    .str.strip()
)

# Boundary diagnostics.
boundary_duplicate_ids = venues_gdf[
    venues_gdf.duplicated("venue_id", keep=False)
]

boundary_empty_geometry = venues_gdf[
    venues_gdf.geometry.isna() | venues_gdf.geometry.is_empty
]

boundary_invalid_geometry = venues_gdf[
    venues_gdf.geometry.notna() & ~venues_gdf.geometry.is_valid
]

# Compare the unique venue ID/name pairs in both files.
boundary_pairs = set(
    venues_gdf[["venue_id", "venue_name"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

lookup_pairs = set(
    venue_geohash9_df[["venue_id", "venue_name"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

boundary_missing_from_lookup = boundary_pairs - lookup_pairs
lookup_missing_from_boundaries = lookup_pairs - boundary_pairs

print(
    f"Boundary venues: "
    f"{venues_gdf[['venue_id', 'venue_name']].drop_duplicates().shape[0]:,}"
)
print(
    f"Lookup venues: "
    f"{venue_geohash9_df[['venue_id', 'venue_name']].drop_duplicates().shape[0]:,}"
)
print(
    f"Duplicate boundary venue IDs: "
    f"{boundary_duplicate_ids['venue_id'].nunique():,}"
)
print(
    f"Missing/empty boundary geometries: "
    f"{len(boundary_empty_geometry):,}"
)
print(
    f"Invalid boundary geometries: "
    f"{len(boundary_invalid_geometry):,}"
)
print(
    f"Boundary venues missing from lookup: "
    f"{len(boundary_missing_from_lookup):,}"
)
print(
    f"Lookup venues missing from boundaries: "
    f"{len(lookup_missing_from_boundaries):,}"
)

if boundary_missing_from_lookup:
    display(
        pd.DataFrame(
            sorted(boundary_missing_from_lookup),
            columns=["venue_id", "venue_name"],
        )
    )

if lookup_missing_from_boundaries:
    display(
        pd.DataFrame(
            sorted(lookup_missing_from_boundaries),
            columns=["venue_id", "venue_name"],
        )
    )

# Confirm that the three named pilot venues exist exactly in the lookup.
pilot_missing = sorted(
    set(PILOT_VENUES) - set(venue_geohash9_df["venue_name"].dropna())
)

if pilot_missing:
    raise ValueError(
        f"Pilot venue names not found exactly in lookup: {pilot_missing}"
    )

Boundary venues: 85
Lookup venues: 85
Duplicate boundary venue IDs: 0
Missing/empty boundary geometries: 0
Invalid boundary geometries: 1
Boundary venues missing from lookup: 0
Lookup venues missing from boundaries: 0


In [10]:
# Venue-geohash diagnostics.
venue_summary_df = (
    venue_geohash9_df.groupby(["venue_id", "venue_name"], as_index=False)
    .agg(
        geohash9_count=("geohash9", "nunique"),
        minimum_coverage_fraction=("coverage_fraction", "min"),
        mean_coverage_fraction=("coverage_fraction", "mean"),
        maximum_coverage_fraction=("coverage_fraction", "max"),
    )
)

overlap_df = (
    venue_geohash9_df.groupby("geohash9", as_index=False)
    .agg(venue_count=("venue_id", "nunique"))
    .query("venue_count > 1")
)

duplicate_assignment_df = venue_geohash9_df[
    venue_geohash9_df.duplicated(["venue_id", "geohash9"], keep=False)
]

print(f"Venue-geohash assignments: {len(venue_geohash9_df):,}")
print(f"Geohash9 cells assigned to multiple venues: {len(overlap_df):,}")
print(f"Duplicate venue-geohash rows: {len(duplicate_assignment_df):,}")
display(venue_summary_df.sort_values("geohash9_count", ascending=False).head(20))
if not overlap_df.empty:
    display(overlap_df.head(20))

Venue-geohash assignments: 9,314
Geohash9 cells assigned to multiple venues: 0
Duplicate venue-geohash rows: 0


,venue_id,venue_name,geohash9_count,minimum_coverage_fraction,mean_coverage_fraction,maximum_coverage_fraction
59,64,Royal Ontario Museum,835,0.8126,0.997121,1.0
54,6,Art Gallery of Ontario,709,0.8075,0.995920,1.0
74,78,Toronto Reference Library,440,0.8272,0.995584,1.0
45,51,Meridian Arts Centre,421,0.8107,0.995347,1.0
33,40,History,388,0.8064,0.991074,1.0
24,32,Four Seasons Centre for the Performing Arts,354,0.8028,0.994053,1.0
4,13,Bentway Staging Grounds,315,0.8185,0.994550,1.0
56,61,Roy Thomson Hall,312,0.8011,0.994966,1.0
83,86,Children's Peace Theatre,297,0.8176,0.997099,1.0
41,48,Malvern Library,292,0.8140,0.994638,1.0


## 6. Upload lookup and diagnostic tables

In [11]:
with timer("Upload Toronto geohash8 lookup"):
    upload_lookup_table(
        toronto8_df,
        TMP_TORONTO_GEOHASH8,
        [("geohash8", "VARCHAR")],
        chunk_size=10000,
    )

with timer("Upload Toronto geohash6 lookup"):
    upload_lookup_table(
        toronto6_df,
        TMP_TORONTO_GEOHASH6,
        [("geohash6", "VARCHAR")],
        chunk_size=10000,
    )

with timer("Upload venue geohash9 lookup"):
    upload_lookup_table(
        venue_geohash9_df[["venue_id", "venue_name", "geohash9", "coverage_fraction"]],
        TMP_VENUE_GEOHASH9,
        [
            ("venue_id", "NUMBER"),
            ("venue_name", "VARCHAR"),
            ("geohash9", "VARCHAR"),
            ("coverage_fraction", "FLOAT"),
        ],
        chunk_size=5000,
    )

[START] Upload Toronto geohash8 lookup


Uploading TMP_TAC_TORONTO_GEOHASH8:   0%|          | 0/122 [00:00<?, ?it/s]

Loaded 1,210,820 rows into DEDICATED.EDDIT_TAC_TMP.TMP_TAC_TORONTO_GEOHASH8
[DONE]  Upload Toronto geohash8 lookup (237.08s)
[START] Upload Toronto geohash6 lookup


Uploading TMP_TAC_TORONTO_GEOHASH6:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 1,307 rows into DEDICATED.EDDIT_TAC_TMP.TMP_TAC_TORONTO_GEOHASH6
[DONE]  Upload Toronto geohash6 lookup (2.95s)
[START] Upload venue geohash9 lookup


Uploading TMP_TAC_VENUE_GEOHASH9:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded 9,314 rows into DEDICATED.EDDIT_TAC_TMP.TMP_TAC_VENUE_GEOHASH9
[DONE]  Upload venue geohash9 lookup (5.80s)


In [12]:
write_geohash_diagnostics_sql = f"""
CREATE OR REPLACE TABLE {OUT_TABLE_GEOHASH_DIAGNOSTICS} AS
WITH per_venue AS (
    SELECT
        venue_id,
        venue_name,
        COUNT(DISTINCT geohash9) AS venue_geohash9_count,
        MIN(coverage_fraction) AS minimum_coverage_fraction,
        AVG(coverage_fraction) AS mean_coverage_fraction,
        MAX(coverage_fraction) AS maximum_coverage_fraction
    FROM {TMP_VENUE_GEOHASH9}
    GROUP BY 1, 2
),
per_cell AS (
    SELECT
        geohash9,
        COUNT(DISTINCT venue_id) AS geohash_venue_count
    FROM {TMP_VENUE_GEOHASH9}
    GROUP BY 1
)
SELECT
    v.venue_id,
    v.venue_name,
    v.geohash9,
    v.coverage_fraction,
    p.venue_geohash9_count,
    p.minimum_coverage_fraction,
    p.mean_coverage_fraction,
    p.maximum_coverage_fraction,
    c.geohash_venue_count,
    IFF(c.geohash_venue_count > 1, TRUE, FALSE) AS overlaps_other_venue
FROM {TMP_VENUE_GEOHASH9} v
INNER JOIN per_venue p USING (venue_id, venue_name)
INNER JOIN per_cell c USING (geohash9)
ORDER BY venue_id, geohash9
"""
with timer(f"Write {OUT_TABLE_GEOHASH_DIAGNOSTICS}"):
    snow_engine.read_sql(write_geohash_diagnostics_sql)

[START] Write DEDICATED.EDDIT_TAC.venues_geohash9_diagnostics
[DONE]  Write DEDICATED.EDDIT_TAC.venues_geohash9_diagnostics (1.15s)


## 7. Pilot run: July–September 2023, three venues

The pilot recreates the main stages at small scale:

- Toronto denominators for the pilot months;
- direct venue geohash9 matching;
- eight-hour raw-stop exclusion;
- deduplication of repeated event-date representations of the same full stop;
- continuous visit sessionization across midnight;
- monthly and half-year device metrics;
- monthly HOME assignment;
- output-like summaries and validation checks.

In [13]:
pilot_venues_sql = sql_str_list(PILOT_VENUES)
print("Pilot venues:", PILOT_VENUES)

Pilot venues: ['Meridian Arts Centre', 'Four Seasons Centre for the Performing Arts', 'Lakeshore Arts']


In [14]:
# Toronto pilot rows retain event-date slices because the Toronto comparison
# denominator is a count of qualifying source stop rows, consistent with the
# legacy denominator. The full-period table will be rebuilt with the same rule.
pilot_toronto_sql = f"""
CREATE OR REPLACE TABLE {PILOT_TORONTO_STOP_ROWS} AS
SELECT
    FLOOR(s.event_date / 100) AS year_month,
    s.cuebiq_id,
    DAYOFWEEKISO(TO_TIMESTAMP_TZ(s.stop_zoned_datetime_in_event_date)) AS dow_iso,
    HOUR(TO_TIMESTAMP_TZ(s.stop_zoned_datetime_in_event_date)) AS hour_of_day
FROM {STOP_BY_EVENT_TABLE} s
INNER JOIN {TMP_TORONTO_GEOHASH8} t
    ON LEFT(s.geohash_id, 8) = t.geohash8
WHERE s.provider_id = '{PROVIDER_ID}'
  AND s.country_code = '{COUNTRY_CODE}'
  AND s.event_date BETWEEN {PILOT_DATE_START} AND {PILOT_DATE_END}
  AND s.dwell_time_minutes IS NOT NULL
  AND s.dwell_time_minutes > 0
  AND s.dwell_time_minutes <= {MAX_RAW_STOP_MINUTES}
  AND s.stop_zoned_datetime_in_event_date IS NOT NULL
"""
with timer("Pilot: build Toronto stop rows"):
    snow_engine.read_sql(pilot_toronto_sql)

pilot_denoms_sql = f"""
CREATE OR REPLACE TABLE {PILOT_TORONTO_DENOMS} AS
WITH periodized AS (
    SELECT year_month, 'all' AS time_period, cuebiq_id FROM {PILOT_TORONTO_STOP_ROWS}
    UNION ALL
    SELECT year_month, 'weekdays', cuebiq_id FROM {PILOT_TORONTO_STOP_ROWS} WHERE dow_iso BETWEEN 1 AND 5
    UNION ALL
    SELECT year_month, 'weekends', cuebiq_id FROM {PILOT_TORONTO_STOP_ROWS} WHERE dow_iso IN (6, 7)
    UNION ALL
    SELECT year_month, 'nine-five', cuebiq_id FROM {PILOT_TORONTO_STOP_ROWS} WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
    UNION ALL
    SELECT year_month, 'evening', cuebiq_id FROM {PILOT_TORONTO_STOP_ROWS} WHERE hour_of_day BETWEEN 17 AND 22
)
SELECT
    year_month,
    time_period,
    COUNT(*) AS total_stop_count,
    COUNT(DISTINCT cuebiq_id) AS total_unique_devices
FROM periodized
GROUP BY 1, 2
"""
with timer("Pilot: build Toronto denominators"):
    snow_engine.read_sql(pilot_denoms_sql)

[START] Pilot: build Toronto stop rows
[DONE]  Pilot: build Toronto stop rows (234.65s)
[START] Pilot: build Toronto denominators
[DONE]  Pilot: build Toronto denominators (9.05s)


In [15]:
# STOP_BY_EVENT_DATE_UPLEVELLED may contain more than one event-date row for a
# stop crossing midnight. The venue table keeps one full-stop representation,
# identified by venue, device, geohash, full start timestamp, and full duration.
pilot_venue_stops_sql = f"""
CREATE OR REPLACE TABLE {PILOT_VENUE_STOP_ROWS} AS
WITH matched AS (
    SELECT
        v.venue_id,
        v.venue_name,
        s.cuebiq_id,
        s.geohash_id AS geohash9,
        TO_TIMESTAMP_TZ(s.stop_zoned_datetime) AS stop_start_ts,
        DATEADD('second', ROUND(s.dwell_time_minutes * 60), TO_TIMESTAMP_TZ(s.stop_zoned_datetime)) AS stop_end_ts,
        s.dwell_time_minutes,
        s.avg_accuracy_meters,
        s.avg_distance_meters,
        s.cluster_size,
        s.event_date,
        s.processing_date,
        ROW_NUMBER() OVER (
            PARTITION BY
                v.venue_id,
                s.cuebiq_id,
                s.geohash_id,
                s.stop_zoned_datetime,
                s.dwell_time_minutes
            ORDER BY s.processing_date DESC, s.event_date ASC
        ) AS rn
    FROM {STOP_BY_EVENT_TABLE} s
    INNER JOIN {TMP_VENUE_GEOHASH9} v
        ON s.geohash_id = v.geohash9
    WHERE s.provider_id = '{PROVIDER_ID}'
      AND s.country_code = '{COUNTRY_CODE}'
      AND s.event_date BETWEEN {PILOT_DATE_START} AND {PILOT_DATE_END}
      AND v.venue_name IN {pilot_venues_sql}
      AND s.stop_zoned_datetime IS NOT NULL
      AND s.dwell_time_minutes IS NOT NULL
      AND s.dwell_time_minutes > 0
      AND s.dwell_time_minutes <= {MAX_RAW_STOP_MINUTES}
)
SELECT
    venue_id,
    venue_name,
    cuebiq_id,
    geohash9,
    stop_start_ts,
    stop_end_ts,
    dwell_time_minutes,
    avg_accuracy_meters,
    avg_distance_meters,
    cluster_size
FROM matched
WHERE rn = 1
  AND TO_NUMBER(TO_CHAR(stop_start_ts, 'YYYYMMDD')) BETWEEN {PILOT_DATE_START} AND {PILOT_DATE_END}
"""
with timer("Pilot: build deduplicated venue stop rows"):
    snow_engine.read_sql(pilot_venue_stops_sql)

[START] Pilot: build deduplicated venue stop rows
[DONE]  Pilot: build deduplicated venue stop rows (229.37s)


In [16]:
# Use the maximum prior stop end, not only LAG(previous end), so nested or
# overlapping stop intervals are sessionized correctly.
pilot_visits_sql = f"""
CREATE OR REPLACE TABLE {PILOT_VENUE_VISITS} AS
WITH ordered AS (
    SELECT
        *,
        MAX(stop_end_ts) OVER (
            PARTITION BY venue_id, cuebiq_id
            ORDER BY stop_start_ts, stop_end_ts, geohash9
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS max_prior_stop_end
    FROM {PILOT_VENUE_STOP_ROWS}
),
flagged AS (
    SELECT
        *,
        IFF(
            max_prior_stop_end IS NULL
            OR stop_start_ts > DATEADD('minute', {VISIT_MERGE_GAP_MINUTES}, max_prior_stop_end),
            1, 0
        ) AS new_visit_flag
    FROM ordered
),
numbered AS (
    SELECT
        *,
        SUM(new_visit_flag) OVER (
            PARTITION BY venue_id, cuebiq_id
            ORDER BY stop_start_ts, stop_end_ts, geohash9
            ROWS UNBOUNDED PRECEDING
        ) AS visit_number
    FROM flagged
),
aggregated AS (
    SELECT
        venue_id,
        venue_name,
        cuebiq_id,
        visit_number,
        MIN(stop_start_ts) AS visit_start_ts,
        MAX(stop_end_ts) AS visit_end_ts,
        DATEDIFF('minute', MIN(stop_start_ts), MAX(stop_end_ts)) AS visit_duration_minutes,
        SUM(dwell_time_minutes) AS summed_raw_dwell_minutes,
        COUNT(*) AS raw_stop_count,
        COUNT(DISTINCT geohash9) AS geohash9_count
    FROM numbered
    GROUP BY 1, 2, 3, 4
)
SELECT
    venue_id,
    venue_name,
    cuebiq_id,
    visit_number,
    visit_start_ts,
    visit_end_ts,
    visit_duration_minutes,
    summed_raw_dwell_minutes,
    raw_stop_count,
    geohash9_count,
    TO_NUMBER(TO_CHAR(visit_start_ts, 'YYYYMMDD')) AS visit_date,
    TO_NUMBER(TO_CHAR(visit_start_ts, 'YYYYMM')) AS year_month,
    CONCAT(YEAR(visit_start_ts), 'H', IFF(MONTH(visit_start_ts) <= 6, '1', '2')) AS half_year,
    DAYOFWEEKISO(visit_start_ts) AS dow_iso,
    HOUR(visit_start_ts) AS hour_of_day
FROM aggregated
WHERE visit_end_ts <= DATEADD('minute', {MAX_MERGED_VISIT_MINUTES}, visit_start_ts)
"""
with timer("Pilot: sessionize venue visits"):
    snow_engine.read_sql(pilot_visits_sql)

[START] Pilot: sessionize venue visits
[DONE]  Pilot: sessionize venue visits (1.57s)


In [17]:
pilot_month_sql = f"""
CREATE OR REPLACE TABLE {PILOT_DEVICE_MONTH_PERIOD} AS
WITH periodized AS (
    SELECT *, 'all' AS time_period FROM {PILOT_VENUE_VISITS}
    UNION ALL
    SELECT *, 'weekdays' FROM {PILOT_VENUE_VISITS} WHERE dow_iso BETWEEN 1 AND 5
    UNION ALL
    SELECT *, 'weekends' FROM {PILOT_VENUE_VISITS} WHERE dow_iso IN (6, 7)
    UNION ALL
    SELECT *, 'nine-five' FROM {PILOT_VENUE_VISITS} WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
    UNION ALL
    SELECT *, 'evening' FROM {PILOT_VENUE_VISITS} WHERE hour_of_day BETWEEN 17 AND 22
)
SELECT
    venue_id,
    venue_name,
    year_month,
    time_period,
    cuebiq_id,
    SUM(raw_stop_count) AS raw_stop_count,
    COUNT(*) AS visit_count,
    COUNT(DISTINCT visit_date) AS distinct_visit_days
FROM periodized
GROUP BY 1, 2, 3, 4, 5
"""
with timer("Pilot: build monthly device-period metrics"):
    snow_engine.read_sql(pilot_month_sql)

pilot_half_sql = f"""
CREATE OR REPLACE TABLE {PILOT_DEVICE_HALF_PERIOD} AS
WITH periodized AS (
    SELECT *, 'all' AS time_period FROM {PILOT_VENUE_VISITS}
    UNION ALL
    SELECT *, 'weekdays' FROM {PILOT_VENUE_VISITS} WHERE dow_iso BETWEEN 1 AND 5
    UNION ALL
    SELECT *, 'weekends' FROM {PILOT_VENUE_VISITS} WHERE dow_iso IN (6, 7)
    UNION ALL
    SELECT *, 'nine-five' FROM {PILOT_VENUE_VISITS} WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
    UNION ALL
    SELECT *, 'evening' FROM {PILOT_VENUE_VISITS} WHERE hour_of_day BETWEEN 17 AND 22
)
SELECT
    venue_id,
    venue_name,
    half_year,
    time_period,
    cuebiq_id,
    SUM(raw_stop_count) AS raw_stop_count,
    COUNT(*) AS visit_count,
    COUNT(DISTINCT visit_date) AS distinct_visit_days,
    COUNT(DISTINCT year_month) AS active_months
FROM periodized
GROUP BY 1, 2, 3, 4, 5
"""
with timer("Pilot: build half-year device-period metrics"):
    snow_engine.read_sql(pilot_half_sql)

[START] Pilot: build monthly device-period metrics
[DONE]  Pilot: build monthly device-period metrics (1.03s)
[START] Pilot: build half-year device-period metrics
[DONE]  Pilot: build half-year device-period metrics (1.08s)


In [18]:
pilot_home_sql = f"""
CREATE OR REPLACE TABLE {PILOT_HOME_MONTH} AS
WITH needed_device_months AS (
    SELECT DISTINCT cuebiq_id, year_month
    FROM {PILOT_DEVICE_MONTH_PERIOD}
),
ranked AS (
    SELECT
        h.cuebiq_id,
        FLOOR(h.snapshot_event_date / 100) AS year_month,
        h.geohash_id AS home_geohash6,
        h.confidence_level,
        h.snapshot_event_date,
        ROW_NUMBER() OVER (
            PARTITION BY h.cuebiq_id, FLOOR(h.snapshot_event_date / 100)
            ORDER BY h.snapshot_event_date DESC
        ) AS rn
    FROM {DEVICE_RECURRING_AREA_TABLE} h
    INNER JOIN needed_device_months n
        ON h.cuebiq_id = n.cuebiq_id
       AND FLOOR(h.snapshot_event_date / 100) = n.year_month
    INNER JOIN {TMP_TORONTO_GEOHASH6} t
        ON h.geohash_id = t.geohash6
    WHERE h.provider_id = '{PROVIDER_ID}'
      AND h.country_code = '{COUNTRY_CODE}'
      AND h.snapshot_event_date BETWEEN {PILOT_DATE_START} AND {PILOT_DATE_END}
      AND h.tag_type_code = 'HOME'
      AND h.confidence_level >= {HOME_CONFIDENCE_MIN}
)
SELECT cuebiq_id, year_month, home_geohash6, confidence_level, snapshot_event_date
FROM ranked
WHERE rn = 1
"""
with timer("Pilot: build monthly HOME assignments"):
    snow_engine.read_sql(pilot_home_sql)

pilot_home_join_sql = f"""
CREATE OR REPLACE TABLE {PILOT_DEVICE_HOME_MONTH_PERIOD} AS
SELECT
    v.venue_id,
    v.venue_name,
    v.year_month,
    v.time_period,
    h.home_geohash6,
    v.cuebiq_id,
    v.raw_stop_count,
    v.visit_count,
    v.distinct_visit_days
FROM {PILOT_DEVICE_MONTH_PERIOD} v
INNER JOIN {PILOT_HOME_MONTH} h
    ON v.cuebiq_id = h.cuebiq_id
   AND v.year_month = h.year_month
"""
with timer("Pilot: join venue metrics to HOME"):
    snow_engine.read_sql(pilot_home_join_sql)

[START] Pilot: build monthly HOME assignments
[DONE]  Pilot: build monthly HOME assignments (495.12s)
[START] Pilot: join venue metrics to HOME
[DONE]  Pilot: join venue metrics to HOME (1.27s)


In [19]:
pilot_validation_sql = f"""
SELECT 'toronto_stop_rows' AS measure, COUNT(*) AS value FROM {PILOT_TORONTO_STOP_ROWS}
UNION ALL SELECT 'venue_stop_rows', COUNT(*) FROM {PILOT_VENUE_STOP_ROWS}
UNION ALL SELECT 'merged_visits', COUNT(*) FROM {PILOT_VENUE_VISITS}
UNION ALL SELECT 'device_month_period_rows', COUNT(*) FROM {PILOT_DEVICE_MONTH_PERIOD}
UNION ALL SELECT 'device_half_period_rows', COUNT(*) FROM {PILOT_DEVICE_HALF_PERIOD}
UNION ALL SELECT 'home_month_rows', COUNT(*) FROM {PILOT_HOME_MONTH}
UNION ALL SELECT 'device_home_month_period_rows', COUNT(*) FROM {PILOT_DEVICE_HOME_MONTH_PERIOD}
"""
display(snow_engine.read_sql(pilot_validation_sql))

print("Pilot monthly venue metrics:")
display(snow_engine.read_sql(f"""
SELECT
    v.venue_name,
    v.year_month,
    v.time_period,
    SUM(v.raw_stop_count) AS raw_stop_count,
    SUM(v.visit_count) AS visit_count,
    COUNT(DISTINCT v.cuebiq_id) AS unique_devices,
    SUM(v.raw_stop_count) / NULLIF(d.total_stop_count, 0) AS raw_stop_prop,
    COUNT(DISTINCT v.cuebiq_id) / NULLIF(d.total_unique_devices, 0) AS unique_devices_prop
FROM {PILOT_DEVICE_MONTH_PERIOD} v
INNER JOIN {PILOT_TORONTO_DENOMS} d USING (year_month, time_period)
GROUP BY 1, 2, 3, d.total_stop_count, d.total_unique_devices
ORDER BY 1, 2, 3
"""))

print("Pilot repeat-visitor metrics:")
display(snow_engine.read_sql(f"""
SELECT
    venue_name,
    half_year,
    time_period,
    SUM(raw_stop_count) AS raw_stop_count,
    SUM(visit_count) AS visit_count,
    COUNT(*) AS unique_devices,
    COUNT_IF(distinct_visit_days >= 2) AS repeat_devices,
    COUNT_IF(distinct_visit_days >= 2) / NULLIF(COUNT(*), 0) AS repeat_device_prop,
    SUM(visit_count) / NULLIF(COUNT(*), 0) AS visits_per_unique_device
FROM {PILOT_DEVICE_HALF_PERIOD}
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""))

print("Pilot sessionization diagnostics:")
display(snow_engine.read_sql(f"""
SELECT
    venue_name,
    COUNT(*) AS visits,
    SUM(raw_stop_count) AS raw_stops_after_deduplication,
    AVG(raw_stop_count) AS mean_raw_stops_per_visit,
    COUNT_IF(raw_stop_count > 1) AS visits_merging_multiple_stops,
    MAX(visit_duration_minutes) AS maximum_visit_minutes
FROM {PILOT_VENUE_VISITS}
GROUP BY 1
ORDER BY 1
"""))

print("Pilot HOME match rate:")
display(snow_engine.read_sql(f"""
WITH all_devices AS (
    SELECT COUNT(DISTINCT CONCAT(cuebiq_id, '-', year_month)) AS n
    FROM {PILOT_DEVICE_MONTH_PERIOD}
),
matched AS (
    SELECT COUNT(DISTINCT CONCAT(cuebiq_id, '-', year_month)) AS n
    FROM {PILOT_DEVICE_HOME_MONTH_PERIOD}
)
SELECT all_devices.n AS venue_device_months,
       matched.n AS matched_home_device_months,
       matched.n / NULLIF(all_devices.n, 0) AS home_match_rate
FROM all_devices CROSS JOIN matched
"""))

,measure,value
0,toronto_stop_rows,19999107
1,venue_stop_rows,4787
2,merged_visits,4525
3,device_month_period_rows,10409
4,device_half_period_rows,10331
5,home_month_rows,2200
6,device_home_month_period_rows,5869


Pilot monthly venue metrics:


,venue_name,year_month,time_period,raw_stop_count,visit_count,unique_devices,raw_stop_prop,unique_devices_prop
0,Four Seasons Centre for the Performing Arts,202307,all,163,153,113,0.000023,0.000069
1,Four Seasons Centre for the Performing Arts,202307,evening,78,72,59,0.000038,0.000066
2,Four Seasons Centre for the Performing Arts,202307,nine-five,21,21,18,0.000009,0.000023
3,Four Seasons Centre for the Performing Arts,202307,weekdays,96,95,78,0.000019,0.000057
4,Four Seasons Centre for the Performing Arts,202307,weekends,67,58,44,0.000032,0.000055
5,Four Seasons Centre for the Performing Arts,202308,all,100,96,85,0.000014,0.000054
6,Four Seasons Centre for the Performing Arts,202308,evening,44,42,36,0.000022,0.000041
7,Four Seasons Centre for the Performing Arts,202308,nine-five,16,16,15,0.000007,0.000018
8,Four Seasons Centre for the Performing Arts,202308,weekdays,71,67,59,0.000013,0.000042
9,Four Seasons Centre for the Performing Arts,202308,weekends,29,29,27,0.000018,0.000039


Pilot repeat-visitor metrics:


,venue_name,half_year,time_period,raw_stop_count,visit_count,unique_devices,repeat_devices,repeat_device_prop,visits_per_unique_device
0,Four Seasons Centre for the Performing Arts,2023H2,all,370,350,271,21,0.077491,1.291513
1,Four Seasons Centre for the Performing Arts,2023H2,evening,165,155,124,7,0.056452,1.250000
2,Four Seasons Centre for the Performing Arts,2023H2,nine-five,57,56,49,3,0.061224,1.142857
3,Four Seasons Centre for the Performing Arts,2023H2,weekdays,243,236,193,11,0.056995,1.222798
4,Four Seasons Centre for the Performing Arts,2023H2,weekends,127,114,91,5,0.054945,1.252747
5,Lakeshore Arts,2023H2,all,16,16,10,2,0.200000,1.600000
6,Lakeshore Arts,2023H2,evening,8,8,4,2,0.500000,2.000000
7,Lakeshore Arts,2023H2,nine-five,7,7,6,1,0.166667,1.166667
8,Lakeshore Arts,2023H2,weekdays,15,15,9,2,0.222222,1.666667
9,Lakeshore Arts,2023H2,weekends,1,1,1,0,0.000000,1.000000


Pilot sessionization diagnostics:


,venue_name,visits,raw_stops_after_deduplication,mean_raw_stops_per_visit,visits_merging_multiple_stops,maximum_visit_minutes
0,Four Seasons Centre for the Performing Arts,350,370,1.057143,17,475
1,Lakeshore Arts,16,16,1.000000,0,372
2,Meridian Arts Centre,4159,4378,1.052657,150,479


Pilot HOME match rate:


,venue_device_months,matched_home_device_months,home_match_rate
0,3960,2200,0.555556


## 8. Full run

Run the following cells after reviewing the pilot. All affected full-run working tables and final outputs are recreated with `CREATE OR REPLACE TABLE`; the obsolete earlier contents are therefore overwritten without requiring a manual drop.

### Safe full-run optimizations applied

The full run preserves the pilot-tested source filters, geohash joins, stop
deduplication, visit-session rules, period definitions, HOME assignment, and
final output grains.

Only low-risk implementation changes are applied:

- unused stop-quality columns are not persisted in the full venue-stop table;
- monthly and half-year periodization carries only columns needed downstream;
- final CTAS statements do not perform unnecessary physical sorting;
- old tables are replaced atomically with `CREATE OR REPLACE`;
- optional cleanup occurs only after final outputs and exports succeed.

The expensive Toronto, venue, and HOME source scans are otherwise unchanged.


### 8.1 Rebuild Toronto-wide rows and denominators for July 2023–June 2026

In [13]:
build_toronto_stop_rows_sql = f"""
CREATE OR REPLACE TABLE {TMP_TORONTO_STOP_ROWS} AS
SELECT
    FLOOR(s.event_date / 100) AS year_month,
    s.cuebiq_id,
    DAYOFWEEKISO(TO_TIMESTAMP_TZ(s.stop_zoned_datetime_in_event_date)) AS dow_iso,
    HOUR(TO_TIMESTAMP_TZ(s.stop_zoned_datetime_in_event_date)) AS hour_of_day
FROM {STOP_BY_EVENT_TABLE} s
INNER JOIN {TMP_TORONTO_GEOHASH8} t
    ON LEFT(s.geohash_id, 8) = t.geohash8
WHERE s.provider_id = '{PROVIDER_ID}'
  AND s.country_code = '{COUNTRY_CODE}'
  AND s.event_date BETWEEN {DATE_START} AND {DATE_END}
  AND s.dwell_time_minutes IS NOT NULL
  AND s.dwell_time_minutes > 0
  AND s.dwell_time_minutes <= {MAX_RAW_STOP_MINUTES}
  AND s.stop_zoned_datetime_in_event_date IS NOT NULL
"""
with timer("Build full Toronto stop rows"):
    snow_engine.read_sql(build_toronto_stop_rows_sql)
display(snow_engine.read_sql(f"SELECT COUNT(*) AS n FROM {TMP_TORONTO_STOP_ROWS}"))

[START] Build full Toronto stop rows
[DONE]  Build full Toronto stop rows (2,303.23s)


,n
0,363320384


In [14]:
build_toronto_denoms_sql = f"""
CREATE OR REPLACE TABLE {TMP_TORONTO_MONTH_DENOMS} AS
WITH periodized AS (
    SELECT year_month, 'all' AS time_period, cuebiq_id FROM {TMP_TORONTO_STOP_ROWS}
    UNION ALL
    SELECT year_month, 'weekdays', cuebiq_id FROM {TMP_TORONTO_STOP_ROWS} WHERE dow_iso BETWEEN 1 AND 5
    UNION ALL
    SELECT year_month, 'weekends', cuebiq_id FROM {TMP_TORONTO_STOP_ROWS} WHERE dow_iso IN (6, 7)
    UNION ALL
    SELECT year_month, 'nine-five', cuebiq_id FROM {TMP_TORONTO_STOP_ROWS} WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
    UNION ALL
    SELECT year_month, 'evening', cuebiq_id FROM {TMP_TORONTO_STOP_ROWS} WHERE hour_of_day BETWEEN 17 AND 22
)
SELECT
    year_month,
    time_period,
    COUNT(*) AS total_stop_count,
    COUNT(DISTINCT cuebiq_id) AS total_unique_devices
FROM periodized
GROUP BY 1, 2
"""
with timer("Build full Toronto monthly denominators"):
    snow_engine.read_sql(build_toronto_denoms_sql)

display(snow_engine.read_sql(f"""
SELECT * FROM {TMP_TORONTO_MONTH_DENOMS}
ORDER BY year_month, time_period
"""))

[START] Build full Toronto monthly denominators
[DONE]  Build full Toronto monthly denominators (192.22s)


,year_month,time_period,total_stop_count,total_unique_devices
0,202307,all,7133351,1645706
1,202307,evening,2055535,891083
2,202307,nine-five,2266734,785493
3,202307,weekdays,5057156,1366101
4,202307,weekends,2076195,793109
...,...,...,...,...
175,202606,all,13222463,3396121
176,202606,evening,5442566,2500728
177,202606,nine-five,2464202,1018613
178,202606,weekdays,9419817,2857983


### 8.2 Extract and deduplicate venue stops at geohash9

In [15]:
build_venue_stop_rows_sql = f"""
CREATE OR REPLACE TABLE {TMP_VENUE_STOP_ROWS} AS
WITH matched AS (
    SELECT
        v.venue_id,
        v.venue_name,
        s.cuebiq_id,
        s.geohash_id AS geohash9,
        TO_TIMESTAMP_TZ(s.stop_zoned_datetime) AS stop_start_ts,
        DATEADD('second', ROUND(s.dwell_time_minutes * 60), TO_TIMESTAMP_TZ(s.stop_zoned_datetime)) AS stop_end_ts,
        s.dwell_time_minutes,
        s.event_date,
        s.processing_date,
        ROW_NUMBER() OVER (
            PARTITION BY
                v.venue_id,
                s.cuebiq_id,
                s.geohash_id,
                s.stop_zoned_datetime,
                s.dwell_time_minutes
            ORDER BY s.processing_date DESC, s.event_date ASC
        ) AS rn
    FROM {STOP_BY_EVENT_TABLE} s
    INNER JOIN {TMP_VENUE_GEOHASH9} v
        ON s.geohash_id = v.geohash9
    WHERE s.provider_id = '{PROVIDER_ID}'
      AND s.country_code = '{COUNTRY_CODE}'
      AND s.event_date BETWEEN {DATE_START} AND {DATE_END}
      AND s.stop_zoned_datetime IS NOT NULL
      AND s.dwell_time_minutes IS NOT NULL
      AND s.dwell_time_minutes > 0
      AND s.dwell_time_minutes <= {MAX_RAW_STOP_MINUTES}
)
SELECT
    venue_id,
    venue_name,
    cuebiq_id,
    geohash9,
    stop_start_ts,
    stop_end_ts,
    dwell_time_minutes
FROM matched
WHERE rn = 1
  AND TO_NUMBER(TO_CHAR(stop_start_ts, 'YYYYMMDD')) BETWEEN {DATE_START} AND {DATE_END}
"""
with timer("Build full deduplicated venue stop rows"):
    snow_engine.read_sql(build_venue_stop_rows_sql)
display(snow_engine.read_sql(f"SELECT COUNT(*) AS n FROM {TMP_VENUE_STOP_ROWS}"))

[START] Build full deduplicated venue stop rows
[DONE]  Build full deduplicated venue stop rows (2,431.06s)


,n
0,257094


### 8.3 Merge nearby stops into venue visits

In [16]:
build_venue_visits_sql = f"""
CREATE OR REPLACE TABLE {TMP_VENUE_VISITS} AS
WITH ordered AS (
    SELECT
        *,
        MAX(stop_end_ts) OVER (
            PARTITION BY venue_id, cuebiq_id
            ORDER BY stop_start_ts, stop_end_ts, geohash9
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS max_prior_stop_end
    FROM {TMP_VENUE_STOP_ROWS}
),
flagged AS (
    SELECT
        *,
        IFF(
            max_prior_stop_end IS NULL
            OR stop_start_ts > DATEADD('minute', {VISIT_MERGE_GAP_MINUTES}, max_prior_stop_end),
            1, 0
        ) AS new_visit_flag
    FROM ordered
),
numbered AS (
    SELECT
        *,
        SUM(new_visit_flag) OVER (
            PARTITION BY venue_id, cuebiq_id
            ORDER BY stop_start_ts, stop_end_ts, geohash9
            ROWS UNBOUNDED PRECEDING
        ) AS visit_number
    FROM flagged
),
aggregated AS (
    SELECT
        venue_id,
        venue_name,
        cuebiq_id,
        visit_number,
        MIN(stop_start_ts) AS visit_start_ts,
        MAX(stop_end_ts) AS visit_end_ts,
        DATEDIFF('minute', MIN(stop_start_ts), MAX(stop_end_ts)) AS visit_duration_minutes,
        SUM(dwell_time_minutes) AS summed_raw_dwell_minutes,
        COUNT(*) AS raw_stop_count,
        COUNT(DISTINCT geohash9) AS geohash9_count
    FROM numbered
    GROUP BY 1, 2, 3, 4
)
SELECT
    venue_id,
    venue_name,
    cuebiq_id,
    visit_number,
    visit_start_ts,
    visit_end_ts,
    visit_duration_minutes,
    summed_raw_dwell_minutes,
    raw_stop_count,
    geohash9_count,
    TO_NUMBER(TO_CHAR(visit_start_ts, 'YYYYMMDD')) AS visit_date,
    TO_NUMBER(TO_CHAR(visit_start_ts, 'YYYYMM')) AS year_month,
    CONCAT(YEAR(visit_start_ts), 'H', IFF(MONTH(visit_start_ts) <= 6, '1', '2')) AS half_year,
    DAYOFWEEKISO(visit_start_ts) AS dow_iso,
    HOUR(visit_start_ts) AS hour_of_day
FROM aggregated
WHERE visit_end_ts <= DATEADD('minute', {MAX_MERGED_VISIT_MINUTES}, visit_start_ts)
"""
with timer("Build full merged venue visits"):
    snow_engine.read_sql(build_venue_visits_sql)
display(snow_engine.read_sql(f"SELECT COUNT(*) AS n FROM {TMP_VENUE_VISITS}"))

[START] Build full merged venue visits
[DONE]  Build full merged venue visits (3.16s)


,n
0,243573


### 8.4 Build monthly and half-year device metrics

In [17]:
build_device_month_sql = f"""
CREATE OR REPLACE TABLE {TMP_VENUE_DEVICE_MONTH_PERIOD} AS
WITH periodized AS (
    SELECT
        venue_id, venue_name, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'all' AS time_period
    FROM {TMP_VENUE_VISITS}

    UNION ALL

    SELECT
        venue_id, venue_name, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'weekdays'
    FROM {TMP_VENUE_VISITS}
    WHERE dow_iso BETWEEN 1 AND 5

    UNION ALL

    SELECT
        venue_id, venue_name, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'weekends'
    FROM {TMP_VENUE_VISITS}
    WHERE dow_iso IN (6, 7)

    UNION ALL

    SELECT
        venue_id, venue_name, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'nine-five'
    FROM {TMP_VENUE_VISITS}
    WHERE dow_iso BETWEEN 1 AND 5
      AND hour_of_day BETWEEN 9 AND 16

    UNION ALL

    SELECT
        venue_id, venue_name, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'evening'
    FROM {TMP_VENUE_VISITS}
    WHERE hour_of_day BETWEEN 17 AND 22
)
SELECT
    venue_id,
    venue_name,
    year_month,
    time_period,
    cuebiq_id,
    SUM(raw_stop_count) AS raw_stop_count,
    COUNT(*) AS visit_count,
    COUNT(DISTINCT visit_date) AS distinct_visit_days
FROM periodized
GROUP BY 1, 2, 3, 4, 5
"""
with timer("Build monthly venue-device-period metrics"):
    snow_engine.read_sql(build_device_month_sql)


[START] Build monthly venue-device-period metrics
[DONE]  Build monthly venue-device-period metrics (2.81s)


In [18]:
build_device_half_sql = f"""
CREATE OR REPLACE TABLE {TMP_VENUE_DEVICE_HALF_PERIOD} AS
WITH periodized AS (
    SELECT
        venue_id, venue_name, half_year, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'all' AS time_period
    FROM {TMP_VENUE_VISITS}

    UNION ALL

    SELECT
        venue_id, venue_name, half_year, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'weekdays'
    FROM {TMP_VENUE_VISITS}
    WHERE dow_iso BETWEEN 1 AND 5

    UNION ALL

    SELECT
        venue_id, venue_name, half_year, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'weekends'
    FROM {TMP_VENUE_VISITS}
    WHERE dow_iso IN (6, 7)

    UNION ALL

    SELECT
        venue_id, venue_name, half_year, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'nine-five'
    FROM {TMP_VENUE_VISITS}
    WHERE dow_iso BETWEEN 1 AND 5
      AND hour_of_day BETWEEN 9 AND 16

    UNION ALL

    SELECT
        venue_id, venue_name, half_year, year_month, cuebiq_id,
        raw_stop_count, visit_date, 'evening'
    FROM {TMP_VENUE_VISITS}
    WHERE hour_of_day BETWEEN 17 AND 22
)
SELECT
    venue_id,
    venue_name,
    half_year,
    time_period,
    cuebiq_id,
    SUM(raw_stop_count) AS raw_stop_count,
    COUNT(*) AS visit_count,
    COUNT(DISTINCT visit_date) AS distinct_visit_days,
    COUNT(DISTINCT year_month) AS active_months
FROM periodized
GROUP BY 1, 2, 3, 4, 5
"""
with timer("Build half-year venue-device-period metrics"):
    snow_engine.read_sql(build_device_half_sql)


[START] Build half-year venue-device-period metrics
[DONE]  Build half-year venue-device-period metrics (4.21s)


### 8.5 Build monthly Toronto HOME assignments only for needed venue device-months

In [19]:
build_home_month_sql = f"""
CREATE OR REPLACE TABLE {TMP_HOME_MONTH} AS
WITH needed_device_months AS (
    SELECT DISTINCT cuebiq_id, year_month
    FROM {TMP_VENUE_DEVICE_MONTH_PERIOD}
),
ranked AS (
    SELECT
        h.cuebiq_id,
        FLOOR(h.snapshot_event_date / 100) AS year_month,
        h.geohash_id AS home_geohash6,
        h.confidence_level,
        h.snapshot_event_date,
        ROW_NUMBER() OVER (
            PARTITION BY h.cuebiq_id, FLOOR(h.snapshot_event_date / 100)
            ORDER BY h.snapshot_event_date DESC
        ) AS rn
    FROM {DEVICE_RECURRING_AREA_TABLE} h
    INNER JOIN needed_device_months n
        ON h.cuebiq_id = n.cuebiq_id
       AND FLOOR(h.snapshot_event_date / 100) = n.year_month
    INNER JOIN {TMP_TORONTO_GEOHASH6} t
        ON h.geohash_id = t.geohash6
    WHERE h.provider_id = '{PROVIDER_ID}'
      AND h.country_code = '{COUNTRY_CODE}'
      AND h.snapshot_event_date BETWEEN {DATE_START} AND {DATE_END}
      AND h.tag_type_code = 'HOME'
      AND h.confidence_level >= {HOME_CONFIDENCE_MIN}
)
SELECT cuebiq_id, year_month, home_geohash6, confidence_level, snapshot_event_date
FROM ranked
WHERE rn = 1
"""
with timer("Build full monthly HOME assignments"):
    snow_engine.read_sql(build_home_month_sql)
display(snow_engine.read_sql(f"SELECT COUNT(*) AS n FROM {TMP_HOME_MONTH}"))

[START] Build full monthly HOME assignments
[DONE]  Build full monthly HOME assignments (6,763.54s)


,n
0,119650


In [20]:
build_device_home_sql = f"""
CREATE OR REPLACE TABLE {TMP_VENUE_DEVICE_HOME_MONTH_PERIOD} AS
SELECT
    v.venue_id,
    v.venue_name,
    v.year_month,
    v.time_period,
    h.home_geohash6,
    v.cuebiq_id,
    v.raw_stop_count,
    v.visit_count,
    v.distinct_visit_days
FROM {TMP_VENUE_DEVICE_MONTH_PERIOD} v
INNER JOIN {TMP_HOME_MONTH} h
    ON v.cuebiq_id = h.cuebiq_id
   AND v.year_month = h.year_month
"""
with timer("Join full venue metrics to monthly HOME"):
    snow_engine.read_sql(build_device_home_sql)

[START] Join full venue metrics to monthly HOME
[DONE]  Join full venue metrics to monthly HOME (1.90s)


## 9. Write final Snowflake outputs

### 9.1 Monthly venue activity: `venues_stops`

In [21]:
write_venues_stops_sql = f"""
CREATE OR REPLACE TABLE {OUT_TABLE_VENUES_STOPS} AS
WITH venue_month AS (
    SELECT
        venue_id,
        venue_name,
        year_month,
        time_period,
        SUM(raw_stop_count) AS raw_stop_count,
        SUM(visit_count) AS visit_count,
        COUNT(*) AS unique_devices
    FROM {TMP_VENUE_DEVICE_MONTH_PERIOD}
    GROUP BY 1, 2, 3, 4
)
SELECT
    v.venue_id,
    v.venue_name,
    v.year_month,
    v.time_period,
    v.raw_stop_count,
    v.visit_count,
    v.unique_devices,
    v.raw_stop_count / NULLIF(v.unique_devices, 0) AS raw_stops_per_unique_device,
    v.visit_count / NULLIF(v.unique_devices, 0) AS visits_per_unique_device,
    v.raw_stop_count / NULLIF(d.total_stop_count, 0) AS raw_stop_prop,
    v.unique_devices / NULLIF(d.total_unique_devices, 0) AS unique_devices_prop
FROM venue_month v
INNER JOIN {TMP_TORONTO_MONTH_DENOMS} d
    ON v.year_month = d.year_month
   AND v.time_period = d.time_period
"""
with timer(f"Write {OUT_TABLE_VENUES_STOPS}"):
    snow_engine.read_sql(write_venues_stops_sql)

[START] Write DEDICATED.EDDIT_TAC.venues_stops
[DONE]  Write DEDICATED.EDDIT_TAC.venues_stops (1.67s)


### 9.2 Six-month repeat attendance: `venues_repeat_visitors`

In [22]:
write_repeat_sql = f"""
CREATE OR REPLACE TABLE {OUT_TABLE_VENUES_REPEAT} AS
SELECT
    venue_id,
    venue_name,
    half_year,
    CASE half_year
        WHEN '2023H2' THEN 20230701
        WHEN '2024H1' THEN 20240101
        WHEN '2024H2' THEN 20240701
        WHEN '2025H1' THEN 20250101
        WHEN '2025H2' THEN 20250701
        WHEN '2026H1' THEN 20260101
    END AS period_start,
    CASE half_year
        WHEN '2023H2' THEN 20231231
        WHEN '2024H1' THEN 20240630
        WHEN '2024H2' THEN 20241231
        WHEN '2025H1' THEN 20250630
        WHEN '2025H2' THEN 20251231
        WHEN '2026H1' THEN 20260630
    END AS period_end,
    time_period,
    SUM(raw_stop_count) AS raw_stop_count,
    SUM(visit_count) AS visit_count,
    COUNT(*) AS unique_devices,
    COUNT_IF(distinct_visit_days >= 2) AS repeat_devices,
    COUNT_IF(distinct_visit_days >= 2) / NULLIF(COUNT(*), 0) AS repeat_device_prop,
    SUM(visit_count) / NULLIF(COUNT(*), 0) AS visits_per_unique_device,
    COUNT_IF(distinct_visit_days >= 3) AS devices_3plus_visit_days,
    COUNT_IF(active_months >= 2) AS devices_2plus_active_months
FROM {TMP_VENUE_DEVICE_HALF_PERIOD}
GROUP BY 1, 2, 3, 6
"""
with timer(f"Write {OUT_TABLE_VENUES_REPEAT}"):
    snow_engine.read_sql(write_repeat_sql)

[START] Write DEDICATED.EDDIT_TAC.venues_repeat_visitors
[DONE]  Write DEDICATED.EDDIT_TAC.venues_repeat_visitors (1.18s)


### 9.3 Monthly HOME origins: `venues_homes_monthly`

In [23]:
write_homes_monthly_sql = f"""
CREATE OR REPLACE TABLE {OUT_TABLE_VENUES_HOMES_MONTHLY} AS
SELECT
    year_month,
    home_geohash6,
    venue_id,
    venue_name,
    time_period,
    SUM(raw_stop_count) AS raw_stop_count,
    SUM(visit_count) AS visit_count,
    COUNT(*) AS unique_devices
FROM {TMP_VENUE_DEVICE_HOME_MONTH_PERIOD}
GROUP BY 1, 2, 3, 4, 5
"""
with timer(f"Write {OUT_TABLE_VENUES_HOMES_MONTHLY}"):
    snow_engine.read_sql(write_homes_monthly_sql)

[START] Write DEDICATED.EDDIT_TAC.venues_homes_monthly
[DONE]  Write DEDICATED.EDDIT_TAC.venues_homes_monthly (1.32s)


### 9.4 Rolled-up HOME origins: `venues_homes`

In [24]:
write_homes_sql = f"""
CREATE OR REPLACE TABLE {OUT_TABLE_VENUES_HOMES} AS
SELECT
    home_geohash6,
    venue_id,
    venue_name,
    time_period,
    SUM(raw_stop_count) AS raw_stop_count,
    SUM(visit_count) AS visit_count,
    COUNT(DISTINCT cuebiq_id) AS unique_devices
FROM {TMP_VENUE_DEVICE_HOME_MONTH_PERIOD}
GROUP BY 1, 2, 3, 4
"""
with timer(f"Write {OUT_TABLE_VENUES_HOMES}"):
    snow_engine.read_sql(write_homes_sql)

[START] Write DEDICATED.EDDIT_TAC.venues_homes
[DONE]  Write DEDICATED.EDDIT_TAC.venues_homes (1.44s)


### 9.5 Copy Toronto denominators to the final schema

In [25]:
copy_denoms_sql = f"""
CREATE OR REPLACE TABLE {OUT_TABLE_TORONTO_DENOMS} AS
SELECT year_month, time_period, total_stop_count, total_unique_devices
FROM {TMP_TORONTO_MONTH_DENOMS}
"""
with timer(f"Write {OUT_TABLE_TORONTO_DENOMS}"):
    snow_engine.read_sql(copy_denoms_sql)

[START] Write DEDICATED.EDDIT_TAC.toronto_month_denoms
[DONE]  Write DEDICATED.EDDIT_TAC.toronto_month_denoms (0.88s)


## 10. Validation

In [26]:
print("Working-table row counts:")
sanity_sql = f"""
SELECT 'TMP_TAC_TORONTO_STOP_ROWS' AS table_name, COUNT(*) AS n FROM {TMP_TORONTO_STOP_ROWS}
UNION ALL SELECT 'TMP_TAC_TORONTO_MONTH_DENOMS', COUNT(*) FROM {TMP_TORONTO_MONTH_DENOMS}
UNION ALL SELECT 'TMP_TAC_VENUE_STOP_ROWS', COUNT(*) FROM {TMP_VENUE_STOP_ROWS}
UNION ALL SELECT 'TMP_TAC_VENUE_VISITS', COUNT(*) FROM {TMP_VENUE_VISITS}
UNION ALL SELECT 'TMP_TAC_VENUE_DEVICE_MONTH_PERIOD', COUNT(*) FROM {TMP_VENUE_DEVICE_MONTH_PERIOD}
UNION ALL SELECT 'TMP_TAC_VENUE_DEVICE_HALF_PERIOD', COUNT(*) FROM {TMP_VENUE_DEVICE_HALF_PERIOD}
UNION ALL SELECT 'TMP_TAC_HOME_MONTH', COUNT(*) FROM {TMP_HOME_MONTH}
UNION ALL SELECT 'TMP_TAC_VENUE_DEVICE_HOME_MONTH_PERIOD', COUNT(*) FROM {TMP_VENUE_DEVICE_HOME_MONTH_PERIOD}
ORDER BY table_name
"""
display(snow_engine.read_sql(sanity_sql))

Working-table row counts:


,table_name,n
0,TMP_TAC_HOME_MONTH,119650
1,TMP_TAC_TORONTO_MONTH_DENOMS,180
2,TMP_TAC_TORONTO_STOP_ROWS,363320384
3,TMP_TAC_VENUE_DEVICE_HALF_PERIOD,495961
4,TMP_TAC_VENUE_DEVICE_HOME_MONTH_PERIOD,327871
5,TMP_TAC_VENUE_DEVICE_MONTH_PERIOD,536553
6,TMP_TAC_VENUE_STOP_ROWS,257094
7,TMP_TAC_VENUE_VISITS,243573


In [27]:
print("Expected months and denominator completeness:")
display(snow_engine.read_sql(f"""
WITH expected_months AS (
    SELECT TO_NUMBER(TO_CHAR(DATEADD('month', SEQ4(), '2023-07-01'::DATE), 'YYYYMM')) AS year_month
    FROM TABLE(GENERATOR(ROWCOUNT => 36))
),
expected_periods AS (
    SELECT column1 AS time_period
    FROM VALUES ('all'), ('weekdays'), ('weekends'), ('nine-five'), ('evening')
)
SELECT
    e.year_month,
    p.time_period,
    d.total_stop_count,
    d.total_unique_devices,
    IFF(d.year_month IS NULL, TRUE, FALSE) AS missing
FROM expected_months e
CROSS JOIN expected_periods p
LEFT JOIN {OUT_TABLE_TORONTO_DENOMS} d
    ON e.year_month = d.year_month
   AND p.time_period = d.time_period
ORDER BY 1, 2
"""))

Expected months and denominator completeness:


,year_month,time_period,total_stop_count,total_unique_devices,missing
0,202307,all,7133351,1645706,False
1,202307,evening,2055535,891083,False
2,202307,nine-five,2266734,785493,False
3,202307,weekdays,5057156,1366101,False
4,202307,weekends,2076195,793109,False
...,...,...,...,...,...
175,202606,all,13222463,3396121,False
176,202606,evening,5442566,2500728,False
177,202606,nine-five,2464202,1018613,False
178,202606,weekdays,9419817,2857983,False


In [28]:
print("Venue activity validity checks:")
display(snow_engine.read_sql(f"""
SELECT
    COUNT_IF(visit_count > raw_stop_count) AS visit_count_above_raw_stops,
    COUNT_IF(unique_devices > visit_count) AS unique_devices_above_visits,
    COUNT_IF(raw_stop_prop < 0 OR raw_stop_prop > 1) AS invalid_raw_stop_props,
    COUNT_IF(unique_devices_prop < 0 OR unique_devices_prop > 1) AS invalid_unique_device_props,
    MIN(year_month) AS first_month,
    MAX(year_month) AS last_month,
    COUNT(DISTINCT venue_id) AS venues
FROM {OUT_TABLE_VENUES_STOPS}
"""))

print("Repeat-visitor validity checks:")
display(snow_engine.read_sql(f"""
SELECT
    COUNT_IF(repeat_devices > unique_devices) AS repeat_above_unique,
    COUNT_IF(repeat_device_prop < 0 OR repeat_device_prop > 1) AS invalid_repeat_props,
    MIN(half_year) AS first_half_year,
    MAX(half_year) AS last_half_year,
    COUNT(DISTINCT venue_id) AS venues
FROM {OUT_TABLE_VENUES_REPEAT}
"""))

Venue activity validity checks:


,visit_count_above_raw_stops,unique_devices_above_visits,invalid_raw_stop_props,invalid_unique_device_props,first_month,last_month,venues
0,0,0,0,0,202307,202606,85


Repeat-visitor validity checks:


,repeat_above_unique,invalid_repeat_props,first_half_year,last_half_year,venues
0,0,0,2023H2,2026H1,85


In [29]:
print("Sessionization and dwell filtering diagnostics:")
display(snow_engine.read_sql(f"""
SELECT
    venue_name,
    COUNT(*) AS visit_count,
    SUM(raw_stop_count) AS raw_stop_count,
    AVG(raw_stop_count) AS mean_raw_stops_per_visit,
    COUNT_IF(raw_stop_count > 1) AS visits_merging_multiple_stops,
    MAX(visit_duration_minutes) AS maximum_visit_minutes,
    AVG(visit_duration_minutes) AS mean_visit_minutes
FROM {TMP_VENUE_VISITS}
GROUP BY 1
ORDER BY raw_stop_count DESC
"""))

print("HOME match rate by month:")
display(snow_engine.read_sql(f"""
WITH needed AS (
    SELECT year_month, COUNT(DISTINCT cuebiq_id) AS devices
    FROM {TMP_VENUE_DEVICE_MONTH_PERIOD}
    WHERE time_period = 'all'
    GROUP BY 1
),
matched AS (
    SELECT year_month, COUNT(DISTINCT cuebiq_id) AS devices
    FROM {TMP_VENUE_DEVICE_HOME_MONTH_PERIOD}
    WHERE time_period = 'all'
    GROUP BY 1
)
SELECT
    n.year_month,
    n.devices AS venue_devices,
    COALESCE(m.devices, 0) AS home_matched_devices,
    COALESCE(m.devices, 0) / NULLIF(n.devices, 0) AS home_match_rate
FROM needed n
LEFT JOIN matched m USING (year_month)
ORDER BY 1
"""))

Sessionization and dwell filtering diagnostics:


,venue_name,visit_count,raw_stop_count,mean_raw_stops_per_visit,visits_merging_multiple_stops,maximum_visit_minutes,mean_visit_minutes
0,Meridian Arts Centre,91146,95377,1.046420,2861,480,69.444353
1,Toronto Reference Library,18415,19755,1.072767,1032,480,81.924409
2,Royal Ontario Museum,15961,17513,1.097237,1180,480,114.298665
3,Art Gallery of Ontario,11898,12516,1.051942,522,480,66.047907
4,Meridian Hall,8661,8940,1.032213,225,480,100.394527
...,...,...,...,...,...,...,...
80,Gallery TPW,33,33,1.000000,0,135,29.666667
81,Array Space,30,31,1.033333,1,465,129.433333
82,Bluffs Gallery,26,27,1.038462,1,80,30.269231
83,Cedar Ridge Creative Centre,21,21,1.000000,0,260,31.142857


HOME match rate by month:


,year_month,venue_devices,home_matched_devices,home_match_rate
0,202307,4717,2697,0.571762
1,202308,4179,2540,0.607801
2,202309,3693,2182,0.590848
3,202310,5374,2899,0.539449
4,202311,5173,2787,0.538759
5,202312,5551,3109,0.560079
6,202401,6661,4052,0.608317
7,202402,6210,3917,0.630757
8,202403,5871,3595,0.612332
9,202404,5953,3415,0.573660


In [30]:
# Duplicate checks at each final table's intended grain.
duplicate_checks = {
    OUT_TABLE_VENUES_STOPS: "venue_id, year_month, time_period",
    OUT_TABLE_VENUES_REPEAT: "venue_id, half_year, time_period",
    OUT_TABLE_VENUES_HOMES_MONTHLY: "year_month, home_geohash6, venue_id, time_period",
    OUT_TABLE_VENUES_HOMES: "home_geohash6, venue_id, time_period",
    OUT_TABLE_TORONTO_DENOMS: "year_month, time_period",
}

for table_name, grain in duplicate_checks.items():
    result = snow_engine.read_sql(f"""
        SELECT COUNT(*) AS duplicate_groups
        FROM (
            SELECT {grain}, COUNT(*) AS n
            FROM {table_name}
            GROUP BY {grain}
            HAVING COUNT(*) > 1
        )
    """)
    print(f"{table_name}: {int(result.iloc[0, 0]):,} duplicate groups")

DEDICATED.EDDIT_TAC.venues_stops: 0 duplicate groups
DEDICATED.EDDIT_TAC.venues_repeat_visitors: 0 duplicate groups
DEDICATED.EDDIT_TAC.venues_homes_monthly: 0 duplicate groups
DEDICATED.EDDIT_TAC.venues_homes: 0 duplicate groups
DEDICATED.EDDIT_TAC.toronto_month_denoms: 0 duplicate groups


## 11. Export final tables to `./activity/`

In [31]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_TABLES = {
    OUT_TABLE_VENUES_STOPS: (EXPORT_DIR / "venue_stops.csv", "venue_id, year_month, time_period"),
    OUT_TABLE_VENUES_REPEAT: (EXPORT_DIR / "venues_repeat_visitors.csv", "venue_id, half_year, time_period"),
    OUT_TABLE_VENUES_HOMES_MONTHLY: (EXPORT_DIR / "venues_homes_monthly.csv", "venue_id, year_month, time_period, home_geohash6"),
    OUT_TABLE_VENUES_HOMES: (EXPORT_DIR / "venues_homes.csv", "venue_id, time_period, home_geohash6"),
    OUT_TABLE_TORONTO_DENOMS: (EXPORT_DIR / "toronto_month_denoms.csv", "year_month, time_period"),
    OUT_TABLE_GEOHASH_DIAGNOSTICS: (EXPORT_DIR / "venues_geohash9_diagnostics.csv", "venue_id, geohash9"),
}

for table_name, (out_path, order_by) in tqdm(EXPORT_TABLES.items(), desc="Exporting final tables"):
    export_table_to_csv(table_name, out_path, order_by)

Exporting final tables:   0%|          | 0/6 [00:00<?, ?it/s]

[START] Read DEDICATED.EDDIT_TAC.venues_stops
[DONE]  Read DEDICATED.EDDIT_TAC.venues_stops (2.51s)
Exported 12,973 rows -> activity/venue_stops.csv
[START] Read DEDICATED.EDDIT_TAC.venues_repeat_visitors
[DONE]  Read DEDICATED.EDDIT_TAC.venues_repeat_visitors (0.48s)
Exported 2,487 rows -> activity/venues_repeat_visitors.csv
[START] Read DEDICATED.EDDIT_TAC.venues_homes_monthly
[DONE]  Read DEDICATED.EDDIT_TAC.venues_homes_monthly (1.52s)
Exported 149,021 rows -> activity/venues_homes_monthly.csv
[START] Read DEDICATED.EDDIT_TAC.venues_homes
[DONE]  Read DEDICATED.EDDIT_TAC.venues_homes (0.84s)
Exported 44,642 rows -> activity/venues_homes.csv
[START] Read DEDICATED.EDDIT_TAC.toronto_month_denoms
[DONE]  Read DEDICATED.EDDIT_TAC.toronto_month_denoms (0.33s)
Exported 180 rows -> activity/toronto_month_denoms.csv
[START] Read DEDICATED.EDDIT_TAC.venues_geohash9_diagnostics
[DONE]  Read DEDICATED.EDDIT_TAC.venues_geohash9_diagnostics (0.61s)
Exported 9,314 rows -> activity/venues_geohash

## 12. Optional cleanup after successful export

Do not delete working tables before the full run. `CREATE OR REPLACE TABLE`
replaces the old full-run versions only after each replacement query succeeds,
which is safer than dropping them in advance.

After the final tables have been validated and exported, the cell below can
remove:

- pilot-only working tables;
- the obsolete level-8 venue lookup;
- old `TEST3_*` output tables.

It deliberately does **not** drop the current full-run working tables. Keeping
those tables makes it possible to revise a downstream output or rerun
validation without repeating the expensive source scans.

Set `RUN_OPTIONAL_CLEANUP = True` only after reviewing the list.


In [ ]:
RUN_OPTIONAL_CLEANUP = False

OPTIONAL_CLEANUP_TABLES = [
    # Pilot working tables
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_TORONTO_STOP_ROWS",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_TORONTO_MONTH_DENOMS",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_VENUE_STOP_ROWS",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_VENUE_VISITS",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_DEVICE_MONTH_PERIOD",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_DEVICE_HALF_PERIOD",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_HOME_MONTH",
    f"{TMP_SCHEMA}.TMP_TAC_PILOT_DEVICE_HOME_MONTH_PERIOD",

    # Obsolete lookup from the earlier geohash8 venue workflow
    f"{TMP_SCHEMA}.TMP_TAC_VENUE_GEOHASH8",

    # Old test outputs
    f"{OUT_SCHEMA}.TEST3_VENUES_STOPS",
    f"{OUT_SCHEMA}.TEST3_VENUES_HOMES",
]

if RUN_OPTIONAL_CLEANUP:
    for table_name in tqdm(OPTIONAL_CLEANUP_TABLES, desc="Dropping obsolete tables"):
        snow_engine.read_sql(f"DROP TABLE IF EXISTS {table_name}")
        print(f"Dropped: {table_name}")
else:
    print("Cleanup skipped. Set RUN_OPTIONAL_CLEANUP = True after validating exports.")
    print("Tables proposed for cleanup:")
    for table_name in OPTIONAL_CLEANUP_TABLES:
        print(f"  - {table_name}")


## 13. Interpretation and implementation notes

- The Toronto denominator is rebuilt for exactly **July 2023 through June 2026**, with the same eight-hour full-dwell exclusion applied to source rows.
- `raw_stop_prop` remains a like-for-like legacy-style comparison between venue raw stops and Toronto source stop rows. The main behavioural count is now `visit_count`, after venue sessionization.
- Venue stop extraction uses `STOP_ZONED_DATETIME` and full `DWELL_TIME_MINUTES`. The derived end timestamp allows continuous sessionization across midnight without an expensive Toronto-wide sessionization step.
- A merged visit is assigned to the month and time period of its first stop.
- Time periods overlap by design: every visit appears in `all`, in either `weekdays` or `weekends`, and may additionally appear in `nine-five` or `evening`.
- Venue overlap is allowed. A geohash9 assigned to multiple venues can contribute to each venue; this is exposed in `venues_geohash9_diagnostics`.
- The repeat table uses distinct visit dates within each half-year. It should not be interpreted as identifying the same person across half-years.
- HOME is assigned monthly from the latest qualifying monthly snapshot with `confidence_level >= 0.60`, and only Toronto geohash6 HOME locations are retained.
- `venues_homes_monthly` is the canonical HOME-origin output. The rolled-up `venues_homes` table is convenient but is not a definitive multi-year unique-person count because device IDs and HOME assignments can change.
- `CREATE OR REPLACE TABLE` intentionally overwrites affected old working and final tables. Pilot tables use a separate prefix and can be retained for comparison or dropped later.